# Guided Demo: IO and Profiling Smoke Test

Run this notebook after `00_env_config` has already run in the same Fabric notebook session.

This notebook does not create metadata tables and does not ask for workspace IDs, item IDs, or item names. It uses the existing `CONFIG`, `ENV`, and `FABRIC_CONTEXT` values from `00_env_config`.


In [ ]:
# Fabric notebook prerequisite
# %run 00_env_config


## 1. Confirm existing context

Verify that `00_env_config` created the shared runtime values and print the configured targets used by this demo.


In [ ]:
required_names = ["ENV", "CONFIG", "FABRIC_CONTEXT"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "Run %run 00_env_config before this notebook. Missing: "
        + ", ".join(missing_names)
    )

print(f"ENV: {ENV}")
print("Configured targets:")
for target_name, store in CONFIG.path_config.paths[ENV].items():
    store_kind = getattr(store, "kind", "unknown")
    store_name = getattr(store, "name", target_name)
    print(f"- {target_name}: {store_kind} -> {store_name}")


## 2. Configure only demo controls

These are demo-only controls. Target names should match entries already configured in `00_env_config`.


In [ ]:
UNIFIED_TARGET = "Unified"
WAREHOUSE_TARGET = "Warehouse"
WAREHOUSE_SCHEMA = "dbo"
ROW_COUNT = 1_000_000
PARTITIONS = 16
RUN_LAKEHOUSE_UNIT_TEST = True
RUN_WAREHOUSE_UNIT_TEST = True
USE_UPLOADED_STARTER_FILES = True
LAKEHOUSE_TABLE_NAME = "fabricops_unit_lakehouse_orders"
WAREHOUSE_TABLE_NAME = "fabricops_unit_warehouse_orders"

DEMO_FILE_FOLDER = "fabricops_demo/io_profile"
CSV_RELATIVE_PATH = f"{DEMO_FILE_FOLDER}/orders.csv"
EXCEL_RELATIVE_PATH = f"{DEMO_FILE_FOLDER}/products.xlsx"
PARQUET_RELATIVE_PATH = f"{DEMO_FILE_FOLDER}/customers.parquet"

print("Demo controls ready.")


## 3. Small file based read demo

Read uploaded starter CSV, Excel, and Parquet files from the configured Lakehouse Files area. To regenerate the same samples instead, set `USE_UPLOADED_STARTER_FILES = False`.


In [ ]:
from pathlib import Path
import tempfile

import pandas as pd
from pyspark.sql import functions as F

from fabricops_kit import (
    profile_dataframe,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
)
from fabricops_kit.io.shared import resolve_configured_file_path

from notebookutils import mssparkutils


def configured_file_uri(relative_path: str) -> str:
    """Return the configured Lakehouse Files URI for a demo relative path."""
    _store, _relative, uri = resolve_configured_file_path(
        UNIFIED_TARGET,
        relative_path,
        context=FABRIC_CONTEXT,
    )
    return uri


def copy_local_file_to_configured_lakehouse(local_path: str, relative_path: str) -> str:
    """Copy a local sample file to configured Lakehouse Files."""
    target_uri = configured_file_uri(relative_path)
    parent_uri = target_uri.rsplit("/", 1)[0]
    mssparkutils.fs.mkdirs(parent_uri)
    mssparkutils.fs.cp(f"file:{local_path}", target_uri, True)
    return target_uri


def write_spark_parquet_to_configured_lakehouse(df, relative_path: str) -> str:
    """Write a sample Spark dataframe to configured Lakehouse Files as Parquet."""
    target_uri = configured_file_uri(relative_path)
    df.coalesce(1).write.mode("overwrite").parquet(target_uri)
    return target_uri

if USE_UPLOADED_STARTER_FILES:
    print(
        "Reading uploaded starter files from "
        f"Files/{DEMO_FILE_FOLDER}/. "
        "Set USE_UPLOADED_STARTER_FILES = False to regenerate samples."
    )
else:
    orders_pdf = pd.DataFrame(
        {
            "order_id": [1001, 1002, 1003, 1004, 1005],
            "customer_id": [501, 502, 503, 501, 504],
            "order_date": ["2026-01-05", "2026-01-06", "2026-01-07", "2026-01-08", "2026-01-09"],
            "status": ["shipped", "processing", "shipped", "cancelled", "shipped"],
            "amount": [125.50, 89.99, 42.25, 18.00, 310.75],
            "channel": ["web", "store", "partner", "web", "store"],
        }
    )
    products_pdf = pd.DataFrame(
        {
            "product_id": [2001, 2002, 2003, 2004],
            "product_name": ["Starter Widget", "Analytics Adapter", "Governance Pack", "Legacy Connector"],
            "category": ["Hardware", "Software", "Service", "Hardware"],
            "list_price": [29.99, 149.00, 499.00, 79.50],
            "active": [True, True, True, False],
        }
    )
    customers_df = spark.createDataFrame(
        [
            (501, "Northwind Traders", "Enterprise", "2025-03-15", 0.12),
            (502, "Contoso Retail", "SMB", "2025-04-22", 0.33),
            (503, "Fabrikam Health", "Enterprise", "2025-05-10", 0.21),
            (504, "Adventure Works", "Public Sector", "2025-06-01", 0.44),
        ],
        ["customer_id", "customer_name", "segment", "signup_date", "risk_score"],
    )

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        csv_path = tmp_path / "orders.csv"
        excel_path = tmp_path / "products.xlsx"
        orders_pdf.to_csv(csv_path, index=False)
        products_pdf.to_excel(excel_path, index=False)
        copy_local_file_to_configured_lakehouse(str(csv_path), CSV_RELATIVE_PATH)
        copy_local_file_to_configured_lakehouse(str(excel_path), EXCEL_RELATIVE_PATH)

    write_spark_parquet_to_configured_lakehouse(customers_df, PARQUET_RELATIVE_PATH)

orders_files_df = read_lakehouse_csv(CSV_RELATIVE_PATH, target=UNIFIED_TARGET, context=FABRIC_CONTEXT, inferSchema=True)
products_files_df = read_lakehouse_excel(EXCEL_RELATIVE_PATH, target=UNIFIED_TARGET, context=FABRIC_CONTEXT)
customers_files_df = read_lakehouse_parquet(PARQUET_RELATIVE_PATH, target=UNIFIED_TARGET, context=FABRIC_CONTEXT)

display(orders_files_df)
display(products_files_df)
display(customers_files_df)


## 4. Lakehouse unit test

Generate a Spark dataframe, write it to a configured Lakehouse table, read it back, and assert that the row count matches.


In [ ]:
unit_df = (
    spark.range(0, 1000)
    .withColumnRenamed("id", "order_id")
    .withColumn("batch_id", (F.col("order_id") % F.lit(8)).cast("int"))
    .withColumn("amount", (F.col("order_id") * F.lit(1.25)).cast("double"))
)
expected_unit_count = unit_df.count()

if RUN_LAKEHOUSE_UNIT_TEST:
    write_lakehouse_table(
        unit_df,
        LAKEHOUSE_TABLE_NAME,
        target=UNIFIED_TARGET,
        mode="overwrite",
        repartition_by=[PARTITIONS, "batch_id"],
        context=FABRIC_CONTEXT,
    )
    lakehouse_read_df = read_lakehouse_table(
        LAKEHOUSE_TABLE_NAME,
        target=UNIFIED_TARGET,
        context=FABRIC_CONTEXT,
    )
    lakehouse_count = lakehouse_read_df.count()
    assert lakehouse_count == expected_unit_count, (lakehouse_count, expected_unit_count)
    print(f"Lakehouse PASS: {lakehouse_count:,} rows read back.")
    display(lakehouse_read_df.limit(10))
else:
    lakehouse_read_df = unit_df
    lakehouse_count = expected_unit_count
    print("Lakehouse unit test skipped.")


## 5. Profile from Lakehouse

Profile the dataframe returned by `read_lakehouse_table`.


In [ ]:
lakehouse_profile_df = profile_dataframe(
    lakehouse_read_df,
    LAKEHOUSE_TABLE_NAME,
    config=CONFIG,
)
display(lakehouse_profile_df)


## 6. Warehouse unit test

Write the same Spark dataframe to a configured Warehouse table, read it back, and use a pushed-down `COUNT(*)` query as a second verification.


In [ ]:
if RUN_WAREHOUSE_UNIT_TEST:
    write_warehouse_table(
        unit_df,
        WAREHOUSE_SCHEMA,
        WAREHOUSE_TABLE_NAME,
        target=WAREHOUSE_TARGET,
        mode="overwrite",
        context=FABRIC_CONTEXT,
    )
    warehouse_read_df = read_warehouse_table(
        WAREHOUSE_SCHEMA,
        WAREHOUSE_TABLE_NAME,
        target=WAREHOUSE_TARGET,
        context=FABRIC_CONTEXT,
    )
    warehouse_count = warehouse_read_df.count()
    warehouse_count_query_df = read_warehouse_query(
        f"SELECT COUNT(*) AS row_count FROM {WAREHOUSE_SCHEMA}.{WAREHOUSE_TABLE_NAME}",
        target=WAREHOUSE_TARGET,
        context=FABRIC_CONTEXT,
    )
    warehouse_query_count = warehouse_count_query_df.collect()[0]["row_count"]
    assert warehouse_count == expected_unit_count, (warehouse_count, expected_unit_count)
    assert warehouse_query_count == expected_unit_count, (warehouse_query_count, expected_unit_count)
    print(f"Warehouse PASS: {warehouse_count:,} rows read back.")
    display(warehouse_read_df.limit(10))
    display(warehouse_count_query_df)
else:
    warehouse_read_df = unit_df
    warehouse_count = expected_unit_count
    warehouse_query_count = expected_unit_count
    print("Warehouse unit test skipped.")


## 7. Profile from Warehouse

Profile the dataframe returned by `read_warehouse_table`.


In [ ]:
warehouse_profile_df = profile_dataframe(
    warehouse_read_df,
    WAREHOUSE_TABLE_NAME,
    config=CONFIG,
)
display(warehouse_profile_df)


## 8. Large parallel Spark demo

Spark partitions allow distributed processing. This section generates a larger dataset with `spark.range`, repartitions by `batch_id`, and writes it through the same configured IO pattern.

Keep `ROW_COUNT = 1_000_000` and `PARTITIONS = 16` until the basic run passes. Increase `ROW_COUNT` only after confirming your Fabric capacity can handle the workload.


In [ ]:
large_table_name = f"{LAKEHOUSE_TABLE_NAME}_large"
large_df = (
    spark.range(0, ROW_COUNT, numPartitions=PARTITIONS)
    .withColumnRenamed("id", "order_id")
    .withColumn("batch_id", (F.col("order_id") % F.lit(PARTITIONS)).cast("int"))
    .withColumn("amount", (F.col("order_id") * F.lit(0.01)).cast("double"))
    .repartition(PARTITIONS, "batch_id")
)

print(f"Large dataframe partitions: {large_df.rdd.getNumPartitions()}")
print(f"Large dataframe rows: {large_df.count():,}")

if RUN_LAKEHOUSE_UNIT_TEST:
    write_lakehouse_table(
        large_df,
        large_table_name,
        target=UNIFIED_TARGET,
        mode="overwrite",
        partition_by="batch_id",
        context=FABRIC_CONTEXT,
    )
    large_lakehouse_count = read_lakehouse_table(
        large_table_name,
        target=UNIFIED_TARGET,
        context=FABRIC_CONTEXT,
    ).count()
    assert large_lakehouse_count == ROW_COUNT, (large_lakehouse_count, ROW_COUNT)
    print(f"Large Lakehouse PASS: {large_lakehouse_count:,} rows read back.")
else:
    large_lakehouse_count = None

# Optional Warehouse large write. Enable only after the smaller Warehouse unit test passes.
RUN_LARGE_WAREHOUSE_WRITE = False
if RUN_LARGE_WAREHOUSE_WRITE and RUN_WAREHOUSE_UNIT_TEST:
    write_warehouse_table(
        large_df,
        WAREHOUSE_SCHEMA,
        f"{WAREHOUSE_TABLE_NAME}_large",
        target=WAREHOUSE_TARGET,
        mode="overwrite",
        context=FABRIC_CONTEXT,
    )
    print("Large Warehouse write completed.")


## Final PASS summary


In [ ]:
summary_rows = [
    ("file_csv_read", orders_files_df.count()),
    ("file_excel_read", products_files_df.count()),
    ("file_parquet_read", customers_files_df.count()),
    ("lakehouse_unit_rows", lakehouse_count),
    ("lakehouse_profile_rows", lakehouse_profile_df.count()),
    ("warehouse_unit_rows", warehouse_count),
    ("warehouse_query_count", warehouse_query_count),
    ("warehouse_profile_rows", warehouse_profile_df.count()),
    ("large_lakehouse_rows", large_lakehouse_count if large_lakehouse_count is not None else -1),
]
summary_df = spark.createDataFrame(summary_rows, ["check_name", "observed_value"])
display(summary_df)
print("PASS: IO and profiling smoke test completed.")
